# **Step1:** Load Text

In [1]:
import io

In [2]:
with open('Data.txt','r') as f:
  content = f.read().lower()

print(content)

artificial intelligence has become one of the most transformative technologies of the modern era.
over the past decade, researchers and engineers have worked tirelessly to design systems that can learn from data and improve over time.
the concept of teaching machines to recognize patterns is not new, but the scale at which it is being applied today is unprecedented.
universities across the world are offering specialized courses in machine learning, deep learning, and data science.
students are exploring neural networks, optimization algorithms, and probabilistic reasoning to understand how intelligent systems operate.

in research laboratories, scientists experiment with large datasets to train models that can interpret speech, analyze images, and translate languages.
the progress achieved in natural language processing has enabled machines to communicate more effectively with humans.
chatbots and virtual assistants now assist millions of users daily, answering questions and automating

# **Step2:** Creating Character Vocabulary

In [3]:
chars = sorted(set(content))
vocab_size = len(chars)

char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for i, c in enumerate(chars)}

print("Vocabulary size:", vocab_size)

Vocabulary size: 30


In [4]:
print("Characters: ")
print(chars)
print("\nLength of characters: ")
print(len(chars))

Characters: 
['\n', ' ', ',', '.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Length of characters: 
30


# **Step3:** Creating Input Sequences

### Deciding the sequence length

In [5]:
sequence_length = 40
step = 1

**Meaning:**
- Input = 40 characters

- Output = 1 next character

- Move window by 1 each time

### Creating sequences

In [6]:
inputs = []
targets = []

for i in range(0, len(content) - sequence_length, step):
    inputs.append(content[i:i+sequence_length])
    targets.append(content[i+sequence_length])

In [7]:
print("Number of sequences:", len(inputs))
print("Example input:", inputs[0])
print("Example target:", targets[0])

Number of sequences: 5509
Example input: artificial intelligence has become one o
Example target: f


## What is happening above in the loop:


```
inputs = []
targets = []
```
We initialized two empty arrays, named inputs and targets, where the input arrays stores the sequence of 40 characters and the targets array stores the next character for each sequences


```
for i in range(0, len(content) - sequence_length, step):
```
It iterates through the text and it stops at len(content) - sequence_length to avoid index overflow. Here i represents the starting index of each sequence.


```
inputs.append(content[i:i+sequence_length])
```
It extracts the 40 characters starting form index i and this becomes one training input sample.



```
targets.append(content[i+sequence_length])
```
This select the immediate character after the 40 character input, and this is the most model must predcit


**What This Code Does Overall**

- Creates thousands of training samples.

- Each sample contains:

   *   Input: 40 characters

   *   Output: Next character

- Uses a sliding window approach to generate sequential training data.




# **Step4:** Vectorization(One hot encoding)

## **What we already have**:

- inputs -> list of 40 character sequences
- targets -> list of next characters
- char2idx -> mapping of character to index
- vocab_size
- sequence_length

Now we will convert everything into numerical tensors.

In [8]:
import numpy as np


X = np.zeros((len(inputs), sequence_length, vocab_size), dtype=np.bool_)
y = np.zeros((len(inputs), vocab_size), dtype=np.bool_)

for i, seq in enumerate(inputs):
    for t, char in enumerate(seq):
        X[i, t, char2idx[char]] = 1
    y[i, char2idx[targets[i]]] = 1

print("Shape of X: ", X.shape)
print("Shapoe of y: ", y.shape)

Shape of X:  (5509, 40, 30)
Shapoe of y:  (5509, 30)


## What is happening above in the loop:

```
import numpy as np

X = np.zeros((len(inputs), sequence_length, vocab_size), dtype=np.bool_)
y = np.zeros((len(inputs), vocab_size), dtype=np.bool_)
```

We initialize two NumPy arrays:

- **X** stores the input sequences in one-hot encoded format.
- **y** stores the target characters in one-hot encoded format.

`X` is a **3D array** with shape:

- (Number of sequences, Sequence length, Vocabulary size)

`y` is a **2D array** with shape:

- (Number of sequences, Vocabulary size)

All values are initially set to 0.

---

```
for i, seq in enumerate(inputs):
```

- This loop iterates through each input sequence.
- `i` represents the index of the sequence.
- `seq` represents the 40-character string.

---

```
for t, char in enumerate(seq):
```

- This loop iterates through each character in the sequence.
- `t` represents the position (time step).
- `char` represents the actual character.

---

```
X[i, t, char2idx[char]] = 1
```

- Converts each character into a one-hot encoded vector.
- `char2idx[char]` gives the numerical index of that character.
- That index position is set to 1.
- All other positions remain 0.

---

```
y[i, char2idx[targets[i]]] = 1
```

- One-hot encodes the target character.
- `targets[i]` gives the next character.
- Its corresponding index is set to 1 in `y`.

---

## Overall Purpose

This step converts text into numerical tensors so that:

> Given 40 characters (input), the model learns to predict the next character (output).

The data is now ready to be fed into the RNN and LSTM models.

# **Step5:** Creating training sequence

In [9]:
print("Training input shape:", X.shape)
print("Training target shape:", y.shape)

Training input shape: (5509, 40, 30)
Training target shape: (5509, 30)


# **Step6:** Building RNN model

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_model = Sequential()

rnn_model.add(SimpleRNN(128, input_shape=(sequence_length, vocab_size)))

rnn_model.add(Dense(vocab_size, activation='softmax'))

rnn_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        20,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 30)             │         3,870 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,222 (94.62 KB)

 Trainable params: 24,222 (94.62 KB)

 Non-trainable params: 0 (0.00 B)

## Explanation of the Code

```
rnn_model=Sequential()
```
Creates a linear stack of layers.

---

```
rnn_model.add(SimpleRNN(128, input_shape=(sequence_length, vocab_size)))
```

- 128 -> Number of hidden units (neurons).
- sequence_length -> Number of time steps (40 characters).
- vocab_size -> Size of one-hot encoded character vector.

This layer processes the sequential character input.

---

```
rnn_model.add(Dense(vocab_size, activation='softmax'))
```

- Fully connected output layer.
- Output size equals vocabulary size.
- softmax converts outputs into probability distribution over characters.

The model predicts the probability of each possible next character.

---

```
rnn_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
```

- categorical_crossentropy -> Used for multi-class classification.
- adam -> Optimizer for faster convergence.
- accuracy -> To monitor performance.

---

## Purpose

The RNN model learns to predict the next character given a sequence of 40 characters.

## Training the RNN Model with Early Stopping

In [11]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)
history = rnn_model.fit(
    X,
    y,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.0992 - loss: 3.1207
Epoch 2/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.1588 - loss: 2.8447
Epoch 3/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.2393 - loss: 2.6761
Epoch 4/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3009 - loss: 2.4945
Epoch 5/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3304 - loss: 2.3694
Epoch 6/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3657 - loss: 2.2627
Epoch 7/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3838 - loss: 2.1732
Epoch 8/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3854 - loss: 2.1425
Epoch 9/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.3952 - loss: 2.0626
Epoch 10/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.4118 - loss: 1.9939
Epoch 11/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4276 - loss: 1.9314
Epoch 12/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step

## Explanation

### EarlyStopping

- monitor='loss' -> Observes training loss.
- patience=3 -> Stops training if loss does not improve for 3 consecutive epochs.
- restore_best_weights=True -> Restores the best-performing model weights.

---

## Why Use Early Stopping?

- Prevents unnecessary training.
- Avoids overfitting.
- Saves computation time.
- Automatically finds optimal number of epochs.

In [12]:
import numpy as np

best_epoch = np.argmin(history.history['loss']) + 1
best_loss = np.min(history.history['loss'])

print("Best Epoch:", best_epoch)
print("Best Loss:", best_loss)

Best Epoch: 51
Best Loss: 0.6144667267799377


In [13]:
final_loss, final_acc = rnn_model.evaluate(X, y, verbose=0)
print("Restored Loss:", final_loss)

Restored Loss: 0.5112350583076477


# **Step7:** Building LSTM model

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

lstm_model = Sequential()
lstm_model.add(LSTM(128, input_shape=(sequence_length, vocab_size)))
lstm_model.add(Dense(vocab_size, activation='softmax'))

lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        81,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30)             │         3,870 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,278 (333.12 KB)

 Trainable params: 85,278 (333.12 KB)

 Non-trainable params: 0 (0.00 B)

## Explanation of the Code

```
lstm_model = Sequential()
```
- Creates a linear stack of layers.  
- Layers are added one after another.

---

```
lstm_model.add(
    LSTM(128, input_shape=(sequence_length, vocab_size))
)
```
- **128** -> Number of LSTM units (neurons).
- `sequence_length` -> Number of time steps (40 characters).
- `vocab_size` -> Size of one-hot encoded character vector.

The LSTM layer processes sequential input and maintains memory of long-term dependencies using internal gates.

Unlike SimpleRNN, LSTM can remember important information over longer sequences.

---

```
lstm_model.add(
    Dense(vocab_size, activation='softmax')
)
```

- Fully connected output layer.
- Output size equals the vocabulary size.
- `softmax` converts output values into a probability distribution.

This layer predicts the probability of each possible next character.

---

```
lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
```

- categorical_crossentropy -> Used for multi-class classification.
- adam -> Optimizer for efficient gradient updates.
- accuracy -> Metric to monitor training performance.

---

## Purpose

The LSTM model learns to predict the next character given a sequence of 40 characters, while handling long-term dependencies more effectively than a Simple RNN.

In [15]:
early_stop = EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X,
    y,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - accuracy: 0.0973 - loss: 3.1359
Epoch 2/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 7s 77ms/step - accuracy: 0.1289 - loss: 2.9479
Epoch 3/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - accuracy: 0.1639 - loss: 2.8813
Epoch 4/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 7s 77ms/step - accuracy: 0.2075 - loss: 2.7664
Epoch 5/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - accuracy: 0.2595 - loss: 2.5801
Epoch 6/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.2847 - loss: 2.4659
Epoch 7/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 6s 67ms/step - accuracy: 0.3045 - loss: 2.3823
Epoch 8/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 0.3123 - loss: 2.3114
Epoch 9/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 7s 77ms/step - accuracy: 0.3251 - loss: 2.2747
Epoch 10/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - accuracy: 0.3476 - loss: 2.2208
Epoch 11/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 0.3551 - loss: 2.1700
Epoch 12/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 11s 76ms/st

## Purpose

The LSTM model is trained to predict the next character given a sequence of 40 characters.

Early stopping ensures:
- Efficient training
- Prevention of overfitting
- Automatic selection of the best-performing model weights

In [16]:
import numpy as np

best_epoch = np.argmin(history_lstm.history['loss']) + 1
best_loss = np.min(history_lstm.history['loss'])

print("Best Epoch:", best_epoch)
print("Best Loss:", best_loss)

Best Epoch: 73
Best Loss: 0.09105606377124786


#  **Step 8:** Text Generation Function

In [28]:
import numpy as np
import random

def generate_text(model, length=300):

    start_index = random.randint(0, len(content) - sequence_length - 1)

    while start_index > 0 and content[start_index - 1] != "\n":
        start_index -= 1

    seed = content[start_index:start_index + sequence_length]

    generated_text = seed

    for _ in range(length):

        x_pred = np.zeros((1, sequence_length, vocab_size))

        for t, char in enumerate(seed):
            x_pred[0, t, char2idx[char]] = 1

        prediction = model.predict(x_pred, verbose=0)[0]

        next_index = np.argmax(prediction)
        next_char = idx2char[next_index]

        generated_text += next_char

        seed = seed[1:] + next_char

    return generated_text

## Code Explanation – `generate_text()` Function

### Import Statements
```python
import numpy as np
import random
```
- `numpy` is used to create the one-hot encoded input array.
- `random` is used to pick a random starting position from the training text.

---

### Function Definition
```python
def generate_text(model, length=300):
```
- `model` → Trained RNN or LSTM model.
- `length` → Number of characters to generate.

---

### Select Random Starting Index
```python
start_index = random.randint(0, len(content) - sequence_length - 1)
```
- Picks a random position in the training text.
- Ensures enough space for a full `sequence_length`.

---

### Adjust to Line Start
```python
while start_index > 0 and content[start_index - 1] != "\n":
    start_index -= 1
```
- Moves backward until a newline is found.
- Ensures generation starts at a clean sentence/paragraph boundary.

---

### Extract Seed Sequence
```python
seed = content[start_index:start_index + sequence_length]
generated_text = seed
```
- `seed` → Initial input sequence.
- `generated_text` stores the final output.

---

### Generate Characters Loop
```python
for _ in range(length):
```
- Runs `length` times.
- Generates one character per iteration.

---

### Create One-Hot Input Array
```python
x_pred = np.zeros((1, sequence_length, vocab_size))
```
- Shape: `(batch_size=1, sequence_length, vocab_size)`
- Stores one-hot encoded input.

---

### Convert Seed to One-Hot Encoding
```python
for t, char in enumerate(seed):
    x_pred[0, t, char2idx[char]] = 1
```
- Converts each character in `seed` to one-hot format.
- `char2idx` maps characters to their index.

---

### Predict Next Character
```python
prediction = model.predict(x_pred, verbose=0)[0]
```
- Model outputs probability distribution over vocabulary.
- `[0]` extracts the first (and only) batch result.

---

### Greedy Decoding
```python
next_index = np.argmax(prediction)
next_char = idx2char[next_index]
```
- `argmax` selects the character with highest probability.
- `idx2char` converts index back to character.

---

### Update Output and Seed
```python
generated_text += next_char
seed = seed[1:] + next_char
```
- Appends predicted character to output.
- Updates seed using sliding window:
  - Removes first character
  - Adds new predicted character

---

### Return Generated Text
```python
return generated_text
```
- Returns the final generated sequence.

# Run RNN

In [32]:
print("RNN Generated Text:\n")
print(generate_text(rnn_model, 150))


RNN Generated Text:

smart cities may utilize predictive analytics to oriige trains indintimant ofuthecits rathrer ang ther se rearived toes.
terab iens arabextiof reation arnanconss matificistion thes th nemage


# Run LSTM

In [33]:
print("LSTM Generated Text:\n")
print(generate_text(lstm_model, 150))


LSTM Generated Text:

distributed computing techniques allow massive datasets to be processed efficiently.
hardware accelerators such as graphical processing units significantly reduce training time.
optimization


# Retraining LSTM with dropout

In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

lstm_dropout_model = Sequential()

lstm_dropout_model.add(
    LSTM(
        128,
        dropout=0.2,
        recurrent_dropout=0.2,
        input_shape=(sequence_length, vocab_size)
    )
)

lstm_dropout_model.add(
    Dense(vocab_size, activation='softmax')
)

lstm_dropout_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

lstm_dropout_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 128)            │        81,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30)             │         3,870 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,278 (333.12 KB)

 Trainable params: 85,278 (333.12 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
early_stop = EarlyStopping(
    monitor='loss',
    patience=8,
    restore_best_weights=True
)
history_lstm_dropout = lstm_dropout_model.fit(
    X,
    y,
    epochs=200,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 12s 95ms/step - accuracy: 0.0968 - loss: 3.1355
Epoch 2/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 9s 101ms/step - accuracy: 0.1309 - loss: 2.9590
Epoch 3/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 11s 109ms/step - accuracy: 0.1568 - loss: 2.9047
Epoch 4/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - accuracy: 0.1873 - loss: 2.8343
Epoch 5/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 8s 89ms/step - accuracy: 0.2177 - loss: 2.7482
Epoch 6/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 9s 108ms/step - accuracy: 0.2377 - loss: 2.6414
Epoch 7/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - accuracy: 0.2591 - loss: 2.5325
Epoch 8/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 8s 88ms/step - accuracy: 0.2695 - loss: 2.5021
Epoch 9/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 11s 93ms/step - accuracy: 0.2848 - loss: 2.4300
Epoch 10/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 9s 107ms/step - accuracy: 0.2791 - loss: 2.4486
Epoch 11/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 9s 99ms/step - accuracy: 0.2909 - loss: 2.3776
Epoch 12/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 9

In [39]:
import numpy as np

best_epoch = np.argmin(history_lstm_dropout.history['loss']) + 1
best_loss = np.min(history_lstm_dropout.history['loss'])

print("Best Epoch:", best_epoch)
print("Best Loss:", best_loss)

Best Epoch: 196
Best Loss: 0.824650764465332


In [44]:
print("LSTM Generated Text:\n")
print(generate_text(lstm_dropout_model, 150))

LSTM Generated Text:

smart cities may utilize predictive analytics to optimize part interprete systems can probuce coherent paragraphs are collaboration of artificial intelligence speed.

as artificial intellige


In [41]:
print("LSTM Generated Text:\n")
print(generate_text(lstm_dropout_model, 200))


LSTM Generated Text:

financial institutions deploy machine learning algorithms.
a strong theoretical institutions and neural networks are capable of detecting open significantly to design more efficient models to design more efficient models to design more effi


In [43]:
print("LSTM Generated Text:\n")
print(generate_text(lstm_dropout_model, 300))

LSTM Generated Text:

in research laboratories, scientists experimence in real world applications.

the journey of artificial intelligence speed.

as artificial intelligence in real world application.
convolutional neural networks are capable of detecting open significantly to design more efficient models to design more efficient models to design more efficien


In [45]:
print("LSTM Generated Text(Regular LSTM without dropout):\n")
print(generate_text(lstm_model, 400))

LSTM Generated Text(Regular LSTM without dropout):

in research laboratories, scientists experiment with large datasets to train models that can interpret speech, analyze images, and translate languages.
the progress achieved in natural language processing has enabled machines to communicate more effectively with humans.
chatbots and virtual assistants now assist millions of users daily, answering questions and automating routine tasks.
behind these systems lies complex mathematical comp


In [46]:
print("LSTM Generated Text:\n")
print(generate_text(lstm_dropout_model, 400))

LSTM Generated Text:

text generation models demonstrate the creative pateents and engineers such as graphical intelligence speed.

as artificial intelligence in real world application.
convolutional neural networks are capable of detecting open significantly to design more efficient models to design more efficient models to design more efficient models to design more efficient models to design more efficient models to design more efficient models to design 


In [47]:
print("RNN Generated Text:\n")
print(generate_text(rnn_model, 400))

RNN Generated Text:

manufacturing plants integrate intelligent monteonitnes count ane altogre radestions tore them chapesstime tancimersatre tofignterssore suthes celes and teoficent outo models to plonemactine learnimatisy te seromactins unco medutt ingevero muntrabicaltich ivedeta nim tomples sy he urecerligy romurt cal impromedech he estrems tontion ony res bueg turen or and intiverat sntodeste mrangorstead ons ras ondine toment antones mochininations s


# Final Conclusion and Interpretation

## Model Comparison Overview

Three models were implemented and evaluated for next-character prediction:

1. Simple RNN  
2. LSTM (without dropout)  
3. LSTM (with dropout)

All models were trained on the same dataset and evaluated by generating new text of approximately 400 characters using greedy decoding.

---

## Observations

### 1. Simple RNN

The RNN model was able to learn basic character transitions but struggled with long-term dependencies. Although it produced recognizable words initially, the generated text quickly degraded into distorted and nonsensical words. This behavior confirms that Simple RNNs have difficulty retaining long contextual information due to vanishing gradient problems.

**Conclusion:**  
RNN is not suitable for maintaining long-range coherence in text generation tasks.

---

### 2. LSTM (Without Dropout)

The regular LSTM achieved very low loss and high accuracy during training, indicating strong learning capability. The generated text was grammatically correct, coherent, and stylistically similar to the training data. However, the model showed signs of heavy memorization and overfitting, as it closely resembled training-style paragraphs.

**Conclusion:**  
LSTM significantly improves sequence modeling and maintains long-term dependencies effectively. However, without regularization, it tends to overfit small datasets.

---

### 3. LSTM (With Dropout)

The dropout-regularized LSTM converged more slowly and achieved higher final loss compared to the regular LSTM. However, the generated text remained grammatically structured and stylistically consistent. The model exhibited less memorization but showed some repetition of high-probability phrases due to greedy decoding.

**Conclusion:**  
Dropout reduces overfitting and encourages more generalized learning. Although repetition was observed, the model produced stable and coherent text without severe word corruption.

---

## Overall Interpretation

- RNN struggles with long-term dependencies and produces unstable text.
- LSTM effectively captures contextual information and generates coherent paragraphs.
- Adding dropout reduces memorization and improves generalization.
- Greedy decoding can cause repetition in generated text.

The experimental results clearly demonstrate that LSTM outperforms Simple RNN for sequence modeling tasks. Regularization techniques such as dropout help balance memorization and generalization, improving the robustness of generated text.

---

## Final Statement

The implemented models successfully performed next-character prediction and generated text similar in style to the training dataset. Among the tested models, LSTM provided the best overall performance, confirming its superiority in handling sequential data and long-term dependencies.

---

**If I had to rank the results:**
- Best general understanding model -> Dropout LSTM
- Best looking output -> Overfit LSTM
- Worst stability -> RNN

---